# Process and inspect Lichess eval games

This notebook is a cleaned-up workspace for feature extraction and plotting.
It uses the package in `../src/chess_eval_features` rather than defining all
functions inline.

Use this notebook for small exploratory runs. For large exports, use
`process_data.py`.


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
SRC_DIR = PROJECT_ROOT / "src"
sys.path.insert(0, str(SRC_DIR))

PROJECT_ROOT


## Imports

The package provides four main stages:

1. stream eval games from the compressed PGN,
2. extract raw per-game and per-ply tables,
3. derive conservative eval/material/phase features,
4. plot game diagnostics.


In [ ]:
import pandas as pd

from chess_eval_features.parser import PgnZstParser
from chess_eval_features.extract import build_sequence_tables
from chess_eval_features.features import (
  build_conservative_features,
  merge_features_with_game_metadata,
)
from chess_eval_features.plotting import plot_game_diagnostics
from chess_eval_features.export import process_pgn_to_directory


## Parse a small number of eval games

Keep this small in the notebook. The command-line script is the right tool for
large runs.


In [ ]:
pgn_path = PROJECT_ROOT / "data/raw/lichess_db_standard_rated_2017-05.pgn.zst"
n_eval_games = 10

parser = PgnZstParser(pgn_path)
eval_games, scan_report = parser.parse_first_n_with_eval(n_eval_games)

scan_report


## Build raw/semi-raw sequence tables

`df_games` has one row per game.

`df_plies` has one row per ply and preserves raw SAN, raw comments, raw evals,
and raw clocks where available.


In [ ]:
df_plies, df_games = build_sequence_tables(
  eval_games,
  include_raw_pgn=False,
)

df_games.head()


In [ ]:
df_plies.head(20)


## Derive features

`df_plies_feat` is the per-ply table enriched with eval proxies, material,
board-state information, and soft phase weights.

`df_features` is one row per game with conservative aggregate features.


In [ ]:
df_features, df_plies_feat = build_conservative_features(df_plies)
df_features = merge_features_with_game_metadata(df_features, df_games)

df_features.head()


In [ ]:
df_plies_feat[[
  "game_index",
  "ply",
  "side",
  "san_raw",
  "eval_proxy",
  "white_material",
  "black_material",
  "phase_progress",
  "opening_like_weight",
  "middlegame_like_weight",
  "endgame_like_weight",
  "board_parse_ok",
]].head(40)


## Sanity checks

These quick checks make sure the extraction is behaving as expected.


In [ ]:
df_plies_feat["board_parse_ok"].value_counts(dropna=False)


In [ ]:
eval_density_by_game = (
  df_plies_feat
  .groupby("game_index")
  .agg(
    n_plies=("ply", "count"),
    n_eval=("has_any_eval", "sum"),
    n_mate_eval=("is_mate_eval", "sum"),
  )
  .reset_index()
)

eval_density_by_game["evals_per_ply"] = (
  eval_density_by_game["n_eval"] /
  eval_density_by_game["n_plies"]
)

eval_density_by_game


## Plot one game

Change `game_pos` to flip through games.


In [ ]:
plot_df = plot_game_diagnostics(
  df_plies_feat=df_plies_feat,
  df_games=df_games,
  game_pos=0,
)


In [ ]:
plot_df[
  plot_df[[
    "is_annotated_move",
    "is_castle",
    "queen_lost",
    "is_mate_eval",
  ]].any(axis=1)
][[
  "ply",
  "move_number",
  "side",
  "san_raw",
  "move_annotation",
  "eval_proxy",
  "white_material",
  "black_material",
  "is_castle",
  "white_castled",
  "black_castled",
  "white_queen_lost",
  "black_queen_lost",
  "is_mate_eval",
]]


## Test dummy export

This writes a small dummy export under `data/processed/_dummy_export`.
Use this to check that the export code works before running a large job.


In [ ]:
dummy_output_dir = PROJECT_ROOT / "data/processed/_dummy_export"

manifest = process_pgn_to_directory(
  input_path=pgn_path,
  output_dir=dummy_output_dir,
  n_eval_games=10,
  batch_size=5,
  file_format="csv",
  include_raw_pgn=False,
)

manifest


In [ ]:
list(dummy_output_dir.iterdir())
